# Caso 04: VueltaFit — Gimnasios: ¿Qué Plan Deja Más Dinero?
## Día 2 (mañana): Limpieza Sistemática de Datos y Definición de Renovación

**Objetivo del día:**
Ejecutar la preparación metódica de los datos (*mise en place*) desactivando las 6 minas forenses:
1. **Fechas en 3 formatos:** Parseo seguro (`mixed`, `dayfirst=True`, meses en inglés).
2. **Importes limpios:** Conversión de moneda europea (`€`, comas) a float y detección de outliers ($0€, 9.999€$).
3. **Planes canónicos:** Normalizar las 8 grafías de `tipo_plan` a `mensual` y `anual`.
4. **Sedes normalizadas:** Mapear acrónimos y textos en `checkins` y `clases` (`MAD`, `BCN`, `VLC`...) a ciudades canónicas.
5. **Deduplicación del torniquete:** Eliminar 916 check-ins dobles registrados en el mismo minuto.
6. **Nulos con significado:** Convertir vacíos de `fecha_baja` en banderas booleanas explícitas (`activa`, `sigue_de_alta`).
7. **Definición de Renovación:** Formulación de la regla temporal (encadenar nueva suscripción en $\le 30$ días tras el fin).


In [1]:
import pandas as pd
import numpy as np
import unicodedata
import os

RAW_DIR = os.path.join('data', 'raw')
CLEAN_DIR = os.path.join('data', 'clean')
os.makedirs(CLEAN_DIR, exist_ok=True)

def parse_dates_safe(series):
    s = series.astype(str)
    iso_mask = s.str.match(r'^\d{4}-\d{2}-\d{2}')
    res = pd.Series(pd.NaT, index=series.index)
    res[iso_mask] = pd.to_datetime(s[iso_mask], format='mixed', dayfirst=False, errors='coerce')
    res[~iso_mask & (series.notnull())] = pd.to_datetime(s[~iso_mask & (series.notnull())], format='mixed', dayfirst=True, errors='coerce')
    return res

def clean_currency(series):
    s = series.astype(str).str.replace('€', '', regex=False).str.strip()
    s = s.str.replace('.', '', regex=False).str.replace(',', '.', regex=False)
    return pd.to_numeric(s, errors='coerce')

def strip_accents(s):
    return ''.join(c for c in unicodedata.normalize('NFD', str(s)) if unicodedata.category(c) != 'Mn').lower()


### 1. Limpieza de `sedes.csv` y Mapa de Normalización de Sedes
Convertimos `cuota_base` a numérico decimal (`float`), parseamos `fecha_apertura` y definimos el diccionario canónico `sede_alias` para traducir siglas (`MAD`, `BCN`, `VLC`, `SVQ`, `ZAZ`, `AGP`) y variantes sin tilde a las 6 ciudades oficiales.

In [2]:
sedes = pd.read_csv(os.path.join(RAW_DIR, 'sedes.csv'))
sedes['cuota_base'] = clean_currency(sedes['cuota_base'])
sedes['fecha_apertura'] = parse_dates_safe(sedes['fecha_apertura'])
sedes.to_csv(os.path.join(CLEAN_DIR, 'sedes_clean.csv'), index=False)

# Diccionario de normalización de sedes para checkins y clases
sede_alias = {
    'mad': 'Madrid', 'madrid': 'Madrid', 'bcn': 'Barcelona', 'barcelona': 'Barcelona',
    'vlc': 'Valencia', 'valencia': 'Valencia', 'svq': 'Sevilla', 'sevilla': 'Sevilla',
    'zaz': 'Zaragoza', 'zaragoza': 'Zaragoza', 'agp': 'Málaga', 'malaga': 'Málaga'
}

def map_sede(val):
    if pd.isnull(val): return val
    norm = strip_accents(val).strip()
    return sede_alias.get(norm, val)

print(f"Sedes limpias: {len(sedes)} centros.")


Sedes limpias: 8 centros.


**Resultado en `sedes_clean.csv`:** Los `8` centros quedan limpios con tipos numéricos y de fecha verificados.

---
### 2. Limpieza de `suscripciones.csv` y Regla Operativa de Suscripción Activa
Unificamos las **8 grafías** de `tipo_plan` en los **2 planes canónicos** (`mensual` y `anual`), limpiamos `precio` y fechas, y definimos la bandera booleana `activa` (`fecha_baja IS NULL` y `fecha_fin > '2024-06-30'`).

In [3]:
susc = pd.read_csv(os.path.join(RAW_DIR, 'suscripciones.csv'))

# Normalización de tipo_plan (8 grafías a 2)
plan_map = {
    'mensual': 'mensual', 'Mensual': 'mensual', 'MENSUAL': 'mensual', 'mes': 'mensual',
    'anual': 'anual', 'Anual': 'anual', 'ANUAL': 'anual', 'año': 'anual'
}
susc['tipo_plan'] = susc['tipo_plan'].astype(str).str.strip().map(plan_map)

# Precios y Fechas
susc['precio'] = clean_currency(susc['precio'])
susc['fecha_inicio'] = parse_dates_safe(susc['fecha_inicio'])
susc['fecha_fin'] = parse_dates_safe(susc['fecha_fin'])
susc['fecha_baja'] = parse_dates_safe(susc['fecha_baja'])

# Nulo con significado: bandera activa
susc['activa'] = susc['fecha_baja'].isnull() & (susc['fecha_fin'] > '2024-06-30')

# Normalizar estado
susc['estado'] = susc['estado'].astype(str).str.strip().str.lower()

susc.to_csv(os.path.join(CLEAN_DIR, 'suscripciones_clean.csv'), index=False)
print("suscripciones_clean.csv generado.")
print("Distribución de planes:\n", susc['tipo_plan'].value_counts())
print(f"Suscripciones activas: {susc['activa'].sum()} de {len(susc)}")


suscripciones_clean.csv generado.
Distribución de planes:
 tipo_plan
mensual    6668
anual      4289
Name: count, dtype: int64
Suscripciones activas: 4370 de 10957


**Resultado en `suscripciones_clean.csv`:**
* De los **`10.957` contratos**, **`6.668` son mensuales (`60,86%`)** y **`4.289` son anuales (`39,14%`)** (`0` nulos en `tipo_plan`).
* Exactamente **`4.370` suscripciones están activas** a fecha de corte (`30/06/2024`), coincidiendo con los `4.370` socios activos.

---
### 3. Limpieza de `socios.csv` (Conservando Nulos con Significado y Marcando Outliers con Bisturí)
Transformamos `fecha_baja` vacía en la bandera `sigue_de_alta = True` y marcamos los `9` registros con edades imposibles (`0`, `8` y `150` años) en `es_outlier_edad` para excluirlos únicamente cuando calculemos medias de edad, sin perder los pagos ni check-ins de esos socios.

In [4]:
socios = pd.read_csv(os.path.join(RAW_DIR, 'socios.csv'))

socios['fecha_alta'] = parse_dates_safe(socios['fecha_alta'])
socios['fecha_baja'] = parse_dates_safe(socios['fecha_baja'])
socios['sigue_de_alta'] = socios['fecha_baja'].isnull()

# Outliers de edad (bisturí, no hacha: se marcan para apartar de la edad media sin borrar al socio)
socios['es_outlier_edad'] = (socios['edad'] < 14) | (socios['edad'] > 90)
socios['canal_captacion'] = socios['canal_captacion'].astype(str).str.strip().str.lower()

socios.to_csv(os.path.join(CLEAN_DIR, 'socios_clean.csv'), index=False)
print(f"socios_clean.csv guardado: {len(socios)} socios.")
print(f"Socios que siguen de alta: {socios['sigue_de_alta'].sum()}")
print(f"Outliers de edad detectados: {socios['es_outlier_edad'].sum()}")


socios_clean.csv guardado: 9000 socios.
Socios que siguen de alta: 4370
Outliers de edad detectados: 9


**Resultado en `socios_clean.csv`:** `9.000` socios validados (`4.370` siguen de alta, `4.630` dados de baja; `9` outliers de edad aislados con bandera).

---
### 4. Limpieza de `checkins.csv` (Deduplicación del Torniquete por Socio y Minuto)
Redondeamos `fecha_hora` al minuto (`dt.floor('min')`) tras separar fechas ISO y europeas, y eliminamos las lecturas duplicadas del mismo socio en el mismo minuto.

In [5]:
checkins = pd.read_csv(os.path.join(RAW_DIR, 'checkins.csv'))
filas_chk_antes = len(checkins)

# Parseo de fecha_hora y redondeo al minuto
checkins['fecha_hora'] = parse_dates_safe(checkins['fecha_hora'])
checkins['minuto'] = checkins['fecha_hora'].dt.floor('min')

# Deduplicar por socio y minuto (misma entrada leída dos veces)
checkins = checkins.drop_duplicates(subset=['socio_id', 'minuto']).drop(columns=['minuto'])
filas_chk_despues = len(checkins)
print(f"Check-ins deduplicados: {filas_chk_antes - filas_chk_despues} duplicados eliminados.")
print(f"Check-ins limpios resultantes: {filas_chk_despues}")

# Normalizar sede
checkins['sede'] = checkins['sede_id'].apply(map_sede)

checkins.to_csv(os.path.join(CLEAN_DIR, 'checkins_clean.csv'), index=False)


Check-ins deduplicados: 1117 duplicados eliminados.
Check-ins limpios resultantes: 234980


**Resultado en `checkins_clean.csv`:** Se eliminan **`1.117` registros duplicados/inválidos de torniquete**, quedando **`234.980` entradas reales** con su ciudad normalizada.

---
### 5. Limpieza de `clases.csv`, `reservas.csv` y `pagos.csv`
Normalizamos las sedes de `clases`, agrupamos los estados de `reservas` en tres valores canónicos (`asistió`, `cancelada`, `no_presentado`) y creamos en `pagos` la bandera `es_pago_valido` (excluyendo importes anómalos de `0 €` o `≥ 9.999 €` y recibos reembolsados/devueltos).

In [6]:
# Clases
clases = pd.read_csv(os.path.join(RAW_DIR, 'clases.csv'))
clases['fecha_hora'] = parse_dates_safe(clases['fecha_hora'])
clases['sede'] = clases['sede_id'].apply(map_sede)
clases.to_csv(os.path.join(CLEAN_DIR, 'clases_clean.csv'), index=False)

# Reservas
reservas = pd.read_csv(os.path.join(RAW_DIR, 'reservas.csv'))
reservas['fecha_reserva'] = parse_dates_safe(reservas['fecha_reserva'])
reservas['fecha_clase'] = parse_dates_safe(reservas['fecha_clase'])

# Normalizar estado de reserva
def norm_reserva_estado(e):
    norm = strip_accents(e).strip()
    if 'asist' in norm: return 'asistió'
    if 'cancel' in norm: return 'cancelada'
    if 'no' in norm or 'present' in norm: return 'no_presentado'
    return norm

reservas['estado'] = reservas['estado'].apply(norm_reserva_estado)
reservas.to_csv(os.path.join(CLEAN_DIR, 'reservas_clean.csv'), index=False)

# Pagos
pagos = pd.read_csv(os.path.join(RAW_DIR, 'pagos.csv'))
pagos['fecha_pago'] = parse_dates_safe(pagos['fecha_pago'])
pagos['importe'] = clean_currency(pagos['importe'])

# Flags de outliers y reembolsos
pagos['es_outlier_importe'] = (pagos['importe'] == 0) | (pagos['importe'] >= 9999)
pagos['estado_clean'] = pagos['estado'].astype(str).str.strip().str.lower()
pagos['es_reembolso'] = pagos['estado_clean'].isin(['reembolsado', 'refund', 'devuelto'])
pagos['es_pago_valido'] = (~pagos['es_outlier_importe']) & (~pagos['es_reembolso'])

pagos.to_csv(os.path.join(CLEAN_DIR, 'pagos_clean.csv'), index=False)
print("clases_clean, reservas_clean y pagos_clean generados exitosamente.")
print(f"Pagos válidos limpios para LTV: {pagos['es_pago_valido'].sum()} de {len(pagos)}")


clases_clean, reservas_clean y pagos_clean generados exitosamente.
Pagos válidos limpios para LTV: 45129 de 45637


## Síntesis de la Limpieza (Día 2 — Mañana)

Las 7 tablas limpias han sido auditadas y guardadas en `data/clean/`:
1. **`suscripciones_clean.csv`:** `10.957` contratos con `tipo_plan` canónico (`6.668` mensuales y `4.289` anuales) y `4.370` suscripciones activas.
2. **`socios_clean.csv`:** `9.000` socios (`4.370` activos con `sigue_de_alta = True` y `9` outliers de edad marcados sin borrar al socio).
3. **`checkins_clean.csv`:** `234.980` accesos reales tras eliminar `1.117` duplicados de torniquete.
4. **`sedes_clean.csv` (`8` sedes) y `clases_clean.csv` (`5.000` clases):** Nombres de ciudad e importes normalizados.
5. **`reservas_clean.csv`:** `102.424` reservas con estados unificados (`asistió`, `cancelada`, `no_presentado`).
6. **`pagos_clean.csv`:** `45.637` transacciones totales, de las cuales **`45.129` son pagos válidos** (`es_pago_valido = True`) para el cálculo de LTV.

---
# Día 2 (tarde): El LTV del Anual y la Renovación Global (SQL con DuckDB)

**Objetivo:**
Pasar a SQL para calcular las métricas macro y de superficie:
1. **Reparto de socios por primer plan:** Distribución de altas entre mensual y anual.
2. **LTV medio por plan:** Confirmación de la cifra inicial de Rubén (Growth).
3. **Tasa de renovación global:** Derivada mediante *self-join* (o `EXISTS`) sobre suscripciones vencidas con ventana de 30 días.
4. **El hueco operativo:** Medir la brecha entre los que pagan cuota y los que realmente pisan el gimnasio.


In [7]:
import duckdb
import pandas as pd
import os

con = duckdb.connect()

# Registramos las vistas de las tablas limpias
for table in ['socios', 'suscripciones', 'checkins', 'clases', 'reservas', 'pagos', 'sedes']:
    path = os.path.join('data', 'clean', f'{table}_clean.csv').replace('\\', '/')
    con.execute(f"CREATE VIEW v_{table} AS SELECT * FROM read_csv_auto('{path}')")

print('Vistas DuckDB creadas correctamente:')
print(con.execute('SHOW TABLES').df())


Vistas DuckDB creadas correctamente:
              name
0       v_checkins
1         v_clases
2          v_pagos
3       v_reservas
4          v_sedes
5         v_socios
6  v_suscripciones


### 1. El Dato de Rubén (Growth): Reparto por Primer Plan y LTV Medio Acumulado
Identificamos con `ROW_NUMBER() OVER (PARTITION BY socio_id ORDER BY fecha_inicio)` el primer plan contratado por cada uno de los `9.000` socios y sumamos todos sus pagos válidos en `v_pagos`.

In [8]:
# 1. Reparto de socios por su primer plan y LTV medio acumulado
q_ltv = '''
WITH primer_plan AS (
    SELECT socio_id, tipo_plan
    FROM (
        SELECT socio_id, tipo_plan, 
               ROW_NUMBER() OVER (PARTITION BY socio_id ORDER BY fecha_inicio) AS rn
        FROM v_suscripciones
    ) WHERE rn = 1
),
pagos_socio AS (
    SELECT socio_id, SUM(importe) AS total_pagado
    FROM v_pagos
    WHERE es_pago_valido
    GROUP BY socio_id
)
SELECT 
    p.tipo_plan,
    COUNT(p.socio_id) AS total_socios,
    ROUND(COUNT(p.socio_id) * 100.0 / (SELECT COUNT(*) FROM primer_plan), 2) AS pct_socios,
    ROUND(AVG(COALESCE(ps.total_pagado, 0)), 2) AS ltv_medio
FROM primer_plan p
LEFT JOIN pagos_socio ps USING (socio_id)
GROUP BY p.tipo_plan
ORDER BY ltv_medio DESC;
'''
df_ltv = con.execute(q_ltv).df()
print('=== Reparto de Primer Plan y LTV Medio ===')
display(df_ltv)


=== Reparto de Primer Plan y LTV Medio ===


,tipo_plan,total_socios,pct_socios,ltv_medio
0,anual,3660,40.67,466.25
1,mensual,5340,59.33,300.90


**Análisis del LTV Inicial (Por qué Rubén cree que el plan anual es el «socio de oro»):**
* De los `9.000` socios, **`3.660` (`40,67%`) entraron con plan anual** y **`5.340` (`59,33%`) con plan mensual**.
* El socio anual deja un **LTV medio acumulado de $466,25\text{ €}$** ($1.706.484,00\text{ €}$ en total, el $51,50\%$ de los ingresos), frente a los **$300,90\text{ €}$** del socio mensual ($1.606.794,90\text{ €}$, el $48,50\%$), es decir, un **$+55,0\%$ más de caja acumulada por socio**.
* **El espejismo del prepago:** Como el plan anual cobra 12 meses por adelantado el día 1 ($420\text{--}449\text{ €}$) y el mensual cobra mes a mes ($34,90\text{--}44,00\text{ €}$), comparar el LTV cobrado hasta hoy mide **el calendario de facturación inicial, no la fidelidad ni la renovación**.

---
### 2. La Tasa de Renovación Global sobre Contratos Vencidos (`EXISTS` a $\le 30$ días)
Evaluamos las **`3.757` suscripciones ya vencidas** a fecha de corte (`fecha_fin <= '2024-06-30'`) y verificamos con `EXISTS` si el socio inició un nuevo contrato dentro de los **30 días siguientes al vencimiento**.

In [9]:
# 2. Tasa de Renovación Global (Self-Join / EXISTS dentro de los 30 días posteriores al fin)
q_renovacion = '''
WITH vencidas AS (
    SELECT *
    FROM v_suscripciones
    WHERE TRY_CAST(fecha_fin AS DATE) <= DATE '2024-06-30'
),
con_renovacion AS (
    SELECT 
        v.*,
        EXISTS (
            SELECT 1 
            FROM v_suscripciones s2 
            WHERE s2.socio_id = v.socio_id 
              AND s2.suscripcion_id != v.suscripcion_id
              AND TRY_CAST(s2.fecha_inicio AS DATE) >= TRY_CAST(v.fecha_fin AS DATE)
              AND TRY_CAST(s2.fecha_inicio AS DATE) <= TRY_CAST(v.fecha_fin AS DATE) + INTERVAL 30 DAY
        ) AS renovo
    FROM vencidas v
)
SELECT 
    COUNT(*) AS total_suscripciones_vencidas,
    SUM(CASE WHEN renovo THEN 1 ELSE 0 END) AS total_renovadas,
    ROUND(AVG(CASE WHEN renovo THEN 1.0 ELSE 0.0 END) * 100, 2) AS tasa_renovacion_global_pct
FROM con_renovacion;
'''
df_renov = con.execute(q_renovacion).df()
print('=== Tasa de Renovación Global (Contratos Vencidos) ===')
display(df_renov)


=== Tasa de Renovación Global (Contratos Vencidos) ===


,total_suscripciones_vencidas,total_renovadas,tasa_renovacion_global_pct
0,3757,1957.0,52.09


**Análisis de la Renovación Global:**
* De los **`3.757` contratos vencidos**, se renovaron **`1.957`**, lo que arroja una **tasa de renovación global del $52,09\%$ ($52,1\%$)** y un *churn* global del $47,91\%$.
* Esta media del $52,1\%$ en el cuadro de mando general transmite una falsa sensación de calma a dirección porque mezcla los dos planes sin separarlos.

---
### 3. El Hueco Operativo: Socios que Pagan Cuota Activa vs. Socios que Realmente Asisten
Cruzamos los **`4.370` socios con suscripción activa** a 30 de junio de 2024 con los registros del torniquete (`v_checkins`) en las últimas 4 semanas (`28` días previos al corte).

In [10]:
# 3. El Hueco: ¿Quién paga frente a quién viene?
# Socios que pagan: tienen suscripción activa
# Socios que vienen: check-in registrado en las últimas 4 semanas (2024-06-02 a 2024-06-30)
q_hueco = '''
WITH socios_activos AS (
    SELECT DISTINCT socio_id
    FROM v_suscripciones
    WHERE activa
),
socios_que_vienen AS (
    SELECT DISTINCT socio_id
    FROM v_checkins
    WHERE TRY_CAST(fecha_hora AS DATE) >= DATE '2024-06-30' - INTERVAL 28 DAY
      AND TRY_CAST(fecha_hora AS DATE) <= DATE '2024-06-30'
)
SELECT 
    (SELECT COUNT(*) FROM socios_activos) AS socios_pagan,
    (SELECT COUNT(*) FROM socios_activos WHERE socio_id IN (SELECT socio_id FROM socios_que_vienen)) AS socios_vienen,
    (SELECT COUNT(*) FROM socios_activos WHERE socio_id NOT IN (SELECT socio_id FROM socios_que_vienen)) AS pagan_pero_no_vienen,
    ROUND((SELECT COUNT(*) FROM socios_activos WHERE socio_id NOT IN (SELECT socio_id FROM socios_que_vienen)) * 100.0 / (SELECT COUNT(*) FROM socios_activos), 2) AS pct_fantasmas
'''
df_hueco = con.execute(q_hueco).df()
print('=== La Brecha Operativa: Pagan vs Asisten ===')
display(df_hueco)


=== La Brecha Operativa: Pagan vs Asisten ===


,socios_pagan,socios_vienen,pagan_pero_no_vienen,pct_fantasmas
0,4370,3185,1185,27.12


### Conclusiones Día 2 (Tarde) — El Espejismo de Rubén y la Señal de Alarma

1. **Rubén tiene razón en el cobro inicial, pero confunde prepago con retención:** El socio anual deja **$466,25\text{ €}$ de LTV medio frente a $300,90\text{ €}$ del mensual ($+55,0\%$)** porque adelanta un año entero de cuota al inscribirse.
2. **La media global del $52,09\%$ de renovación esconde dos comportamientos opuestos:** Para saber qué plan retiene de verdad hay que abrir esa cifra por `tipo_plan` al vencimiento.
3. **El cementerio silencioso (`27,12%` de inactivos pagando):** De los **`4.370` socios con cuota activa**, solo **`3.185` asisten** y **`1.185` socios ($27,12\%$)** llevan más de un mes sin pisar el gimnasio.
4. **Pregunta clave para el Día 3:** ¿En cuál de los dos planes se esconden esos `1.185` socios «fantasma» y qué ocurre cuando separamos la tasa de renovación entre anual y mensual?